# TimesNet for Time Series Classification

This notebook demonstrates how to use the TimesNet model for time series classification.

Reference: Paper link: https://openreview.net/pdf?id=ju_Uqw384Oq

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Load and Prepare Data

In [ ]:
def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset for time series classification.

    Args:
        file_path: Path to the .npy file containing the dataset

    Returns:
        Tuple containing (X_train, y_train, X_test, y_test)
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    return X_train, y_train, X_test, y_test

In [ ]:
# Specify the path to your dataset
dataset_path = "../data/raw/your_dataset.npy"  # Change this to your dataset path

# Load the dataset
X_train, y_train, X_test, y_test = load_npy_dataset(dataset_path)

In [ ]:
# Prepare data for TimesNet (format: batch, length, channels)
def prepare_data_for_timesnet(X_train, X_test):
    # Check if data needs reshaping
    if len(X_train.shape) == 2:
        # For univariate time series: reshape from (batch, time) to (batch, time, 1)
        X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # If data has the shape (batch, channels, time), transpose to (batch, time, channels)
    elif len(X_train.shape) == 3 and X_train.shape[1] < X_train.shape[2]:
        X_train = np.transpose(X_train, (0, 2, 1))
        X_test = np.transpose(X_test, (0, 2, 1))

    return X_train, X_test


# Prepare data
X_train, X_test = prepare_data_for_timesnet(X_train, X_test)
print(f"Prepared data shape - X_train: {X_train.shape}, X_test: {X_test.shape}")

## 2. Define the TimeSeriesDataset

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_mark = torch.ones(self.X[idx].shape[0])  # TimesNet needs x_mark
        return self.X[idx], x_mark, self.y[idx]

In [ ]:
# Split training data for validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_split.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## 3. Implement TimesNet Components

We'll define the essential components for the TimesNet model.

In [ ]:
# 1. Inception Block
class Inception_Block_V1(nn.Module):
    def __init__(self, in_channels, out_channels, num_kernels=6, init_weight=True):
        super(Inception_Block_V1, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_kernels = num_kernels
        kernels = []
        for i in range(self.num_kernels):
            kernels.append(
                nn.Conv2d(in_channels, out_channels, kernel_size=2 * i + 1, padding=i)
            )
        self.kernels = nn.ModuleList(kernels)
        if init_weight:
            self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        res_list = []
        for i in range(self.num_kernels):
            res_list.append(self.kernels[i](x))
        res = torch.stack(res_list, dim=-1).mean(-1)
        return res

In [ ]:
# 2. Embedding Layer
import math


class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEmbedding, self).__init__()
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model).float()
        pe.require_grad = False

        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = (
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        ).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return self.pe[:, : x.size(1)]


class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if torch.__version__ >= "1.5.0" else 2
        self.tokenConv = nn.Conv1d(
            in_channels=c_in,
            out_channels=d_model,
            kernel_size=3,
            padding=padding,
            padding_mode="circular",
            bias=False,
        )
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(
                    m.weight, mode="fan_in", nonlinearity="leaky_relu"
                )

    def forward(self, x):
        x = self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2)
        return x


class DataEmbedding(nn.Module):
    def __init__(self, c_in, d_model, dropout=0.1):
        super(DataEmbedding, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(d_model=d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        if x_mark is None:
            x = self.value_embedding(x) + self.position_embedding(x)
        else:
            x = self.value_embedding(x) + self.position_embedding(x)
        return self.dropout(x)

In [ ]:
# 3. TimesBlock and Model
class TimesBlock(nn.Module):
    def __init__(self, configs):
        super(TimesBlock, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.k = configs.top_k
        # parameter-efficient design
        self.conv = nn.Sequential(
            Inception_Block_V1(
                configs.d_model, configs.d_ff, num_kernels=configs.num_kernels
            ),
            nn.GELU(),
            Inception_Block_V1(
                configs.d_ff, configs.d_model, num_kernels=configs.num_kernels
            ),
        )

    def FFT_for_Period(self, x, k=2):
        # [B, T, C]
        xf = torch.fft.rfft(x, dim=1)
        # find period by amplitudes
        frequency_list = abs(xf).mean(0).mean(-1)
        frequency_list[0] = 0
        _, top_list = torch.topk(frequency_list, k)
        top_list = top_list.detach().cpu().numpy()
        period = x.shape[1] // top_list
        return period, abs(xf).mean(-1)[:, top_list]

    def forward(self, x):
        B, T, N = x.size()
        period_list, period_weight = self.FFT_for_Period(x, self.k)

        res = []
        for i in range(self.k):
            period = period_list[i]
            # padding
            if (self.seq_len + self.pred_len) % period != 0:
                length = (((self.seq_len + self.pred_len) // period) + 1) * period
                padding = torch.zeros(
                    [x.shape[0], (length - (self.seq_len + self.pred_len)), x.shape[2]]
                ).to(x.device)
                out = torch.cat([x, padding], dim=1)
            else:
                length = self.seq_len + self.pred_len
                out = x
            # reshape
            out = (
                out.reshape(B, length // period, period, N)
                .permute(0, 3, 1, 2)
                .contiguous()
            )
            # 2D conv: from 1d Variation to 2d Variation
            out = self.conv(out)
            # reshape back
            out = out.permute(0, 2, 3, 1).reshape(B, -1, N)
            res.append(out[:, : (self.seq_len + self.pred_len), :])
        res = torch.stack(res, dim=-1)
        # adaptive aggregation
        period_weight = F.softmax(period_weight, dim=1)
        period_weight = period_weight.unsqueeze(1).unsqueeze(1).repeat(1, T, N, 1)
        res = torch.sum(res * period_weight, -1)
        # residual connection
        res = res + x
        return res


class Config:
    def __init__(self, seq_len, num_class, enc_in):
        self.task_name = "classification"
        self.seq_len = seq_len
        self.label_len = 0
        self.pred_len = 0
        self.num_class = num_class
        self.enc_in = enc_in
        self.d_model = 64
        self.d_ff = 64
        self.dropout = 0.1
        self.e_layers = 2
        self.top_k = 3
        self.num_kernels = 6
        self.embed = "fixed"
        self.freq = "h"
        self.c_out = enc_in


class Model(nn.Module):
    def __init__(self, configs):
        super(Model, self).__init__()
        self.configs = configs
        self.task_name = configs.task_name
        self.seq_len = configs.seq_len
        self.label_len = configs.label_len
        self.pred_len = configs.pred_len
        self.model = nn.ModuleList(
            [TimesBlock(configs) for _ in range(configs.e_layers)]
        )
        self.enc_embedding = DataEmbedding(
            configs.enc_in, configs.d_model, configs.dropout
        )
        self.layer = configs.e_layers
        self.layer_norm = nn.LayerNorm(configs.d_model)

        # For classification task
        self.act = F.gelu
        self.dropout = nn.Dropout(configs.dropout)
        self.projection = nn.Linear(
            configs.d_model * configs.seq_len, configs.num_class
        )

    def classification(self, x_enc, x_mark_enc):
        # embedding
        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))

        # Output
        # the output transformer encoder/decoder embeddings don't include non-linearity
        output = self.act(enc_out)
        output = self.dropout(output)
        # zero-out padding embeddings
        output = output * x_mark_enc.unsqueeze(-1)
        # (batch_size, seq_length * d_model)
        output = output.reshape(output.shape[0], -1)
        output = self.projection(output)  # (batch_size, num_classes)
        return output

    def forward(self, x_enc, x_mark_enc, x_dec=None, x_mark_dec=None):
        if self.task_name == "classification":
            dec_out = self.classification(x_enc, x_mark_enc)
            return dec_out  # [B, N]
        return None

## 4. Create Datasets and DataLoaders

In [ ]:
# Create datasets
train_dataset = TimeSeriesDataset(X_train_split, y_train_split)
val_dataset = TimeSeriesDataset(X_val, y_val)
test_dataset = TimeSeriesDataset(X_test, y_test)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

## 5. Initialize and Configure Model

In [ ]:
# Set device (GPU if available, otherwise CPU)
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
# Get dimensions from data
seq_len = X_train.shape[1]
enc_in = X_train.shape[2]
num_class = len(np.unique(y_train))

# Create model config
config = Config(seq_len=seq_len, num_class=num_class, enc_in=enc_in)

# Initialize model
model = Model(config).to(device)


# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"Model parameters: {count_parameters(model):,}")

## 6. Train the Model

In [ ]:
def train_timesnet(
    model, train_loader, val_loader, num_epochs=100, patience=10, learning_rate=0.001
):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0
    epochs_no_improve = 0
    best_model_path = "timesnet_best_model.pth"

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_times": [],
    }

    start_train_time = time.time()

    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_x, batch_x_mark, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_x_mark = batch_x_mark.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x, batch_x_mark)
            loss = criterion(outputs, batch_y)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_x, batch_x_mark, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_x_mark = batch_x_mark.to(device)
                batch_y = batch_y.to(device)

                outputs = model(batch_x, batch_x_mark)
                loss = criterion(outputs, batch_y)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()

        # Calculate metrics
        train_loss /= len(train_loader)
        train_acc = 100.0 * train_correct / train_total
        val_loss /= len(val_loader)
        val_acc = 100.0 * val_correct / val_total
        epoch_time = time.time() - epoch_start_time

        # Store history
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["epoch_times"].append(epoch_time)

        # Print progress
        print(f"Epoch [{epoch + 1}/{num_epochs}] - Time: {epoch_time:.2f}s")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Early stopping logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved with validation accuracy: {val_acc:.2f}%")
        else:
            epochs_no_improve += 1
            print(f"Epochs without improvement: {epochs_no_improve}/{patience}")

            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch + 1} epochs")
                break

    total_train_time = time.time() - start_train_time
    print(f"Total training time: {total_train_time:.2f} seconds")

    # Load the best model
    model.load_state_dict(torch.load(best_model_path))

    return model, history, total_train_time

In [ ]:
# Train the model
num_epochs = 100
patience = 10
learning_rate = 0.001

trained_model, training_history, total_training_time = train_timesnet(
    model, train_loader, val_loader, num_epochs, patience, learning_rate
)

# 7. Evaluate the model

In [ ]:
model.eval()
test_loss = 0
test_correct = 0
test_total = 0

criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for batch_x, batch_x_mark, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_x_mark = batch_x_mark.to(device)
        batch_y = batch_y.to(device)

        outputs = model(batch_x, batch_x_mark)
        loss = criterion(outputs, batch_y)

        test_loss += loss.item()
        _, predicted = outputs.max(1)
        test_total += batch_y.size(0)
        test_correct += (predicted == batch_y).sum().item()

test_loss /= len(test_loader)
test_acc = 100.0 * test_correct / test_total

print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}%")

# 8. Generate classification report

In [ ]:
y_true = []
y_pred = []

with torch.no_grad():
    for batch_x, batch_x_mark, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_x_mark = batch_x_mark.to(device)
        batch_y = batch_y.to(device)

        outputs = model(batch_x, batch_x_mark)
        _, predicted = outputs.max(1)

        y_true.extend(batch_y.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print("Classification Report:")
print(classification_report(y_true, y_pred))


# 9. Plot confusion matrix

In [ ]:
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
plt.imshow(conf_matrix, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()